# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Method choice: Decision Tree Classifier, with Logistic Regression as a comparison point.

Why: My task (from ML-03) is a scoring/ranking problem — I need a continuous score to rank pages by underperformance risk, not just a binary label. A Decision Tree gives interpretable splits (readable rules, matching the "explainable" theme from Week 2's notebook), while Logistic Regression gives a well-calibrated probability score, useful for ranking via predict_proba. I'm avoiding Random Forest/Gradient Boosting for the main model since my baseline is already a simple rule — a single Decision Tree is the fairest "next step up" comparison, not an unnecessarily complex jump. I'll report both, but treat the Decision Tree as primary since it's most directly comparable to the leakage-trap and baseline notebooks I already built.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks * 1.0 / gsc_impressions as ctr,
    gsc_sum_position * 1.0 / gsc_impressions as avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

df['log_impressions'] = np.log1p(df['gsc_impressions'])
df['position_tier_ctr'] = df['avg_position'].apply(lambda p: 0.0039 if p <= 10 else 0.0018)
df['underperforming'] = (df['ctr'] < df['position_tier_ctr']).astype(int)

# GROUPED split by client_hash_id — no client appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")
print(f"Client overlap between train/test: {len(set(train_df['client_hash_id']) & set(test_df['client_hash_id']))}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 2690999 rows, 37 clients
Test: 920062 rows, 10 clients
Client overlap between train/test: 0


Split design: I used GroupShuffleSplit grouped by client_hash_id, not a random row-level split. Result: 2,690,999 training rows across 37 clients, 920,062 test rows across 10 clients, with zero client overlap confirmed. This ensures no client appears in both train and test — critical given the "unbalanced history" limitation flagged in ML-04/ML-05: a random row-level split could let the model learn client-specific quirks and appear to generalize well when it's really just memorizing individual clients.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

features = ['log_impressions', 'avg_position']
X_train, y_train = train_df[features], train_df['underperforming']
X_test, y_test = test_df[features], test_df['underperforming']

# Decision Tree (primary)
tree = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

# Logistic Regression (comparison)
logreg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)

# BASELINE: the rule-based score from ML-07 (thresholded at 0.001 on the score gap)
test_df['baseline_score'] = test_df['position_tier_ctr'] - test_df['ctr']
baseline_pred = (test_df['baseline_score'] > 0.001).astype(int)

def get_metrics(y_true, y_pred, name):
    return {
        'method': name,
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred)
    }

results = pd.DataFrame([
    get_metrics(y_test, baseline_pred, 'Baseline (ML-07 rule)'),
    get_metrics(y_test, tree_pred, 'Decision Tree'),
    get_metrics(y_test, logreg_pred, 'Logistic Regression'),
])

results


,method,precision,recall,f1
0,Baseline (ML-07 rule),1.000000,0.989007,0.994473
1,Decision Tree,0.977980,0.702657,0.817767
2,Logistic Regression,0.979027,0.699362,0.815894


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Model vs. baseline comparison — interpretation:

The baseline (precision=1.00, recall=0.99, f1=0.99) dramatically outperforms both trained models. This is not because the rule is genuinely smarter — it's because my baseline score and my label were both derived from the same underlying comparison (ctr vs. position_tier_ctr), just at slightly different thresholds. The baseline is therefore close to restating its own label, which explains the near-perfect score. This is an important honest catch: comparing a rule-derived label against that same rule as a "model" isn't a fair contest.

The Decision Tree (precision=0.98, recall=0.70, f1=0.82) and Logistic Regression (precision=0.98, recall=0.70, f1=0.82) perform almost identically to each other, and both show high precision but meaningfully lower recall than the baseline. This means: when these models do flag a page as underperforming, they're usually right (98% precision) — but they miss about 30% of true underperformers (recall=0.70), likely because log_impressions and avg_position alone don't fully capture the CTR signal the label was built from.

What this means practically: this comparison is not a clean "model beats baseline" story like Notebook 1's original demo. The honest conclusion is that my label definition is too tightly coupled to my baseline rule to serve as a fair benchmark — the real value of the trained models here is that they generalize using only position and volume, without needing ctr directly, which could matter for predicting on future data where ctr for that period isn't naturally known yet at decision time (whereas today's ctr and today's rule are trivially the same thing).

A fairer future baseline would use a fixed, universal rule not derived from the exact same threshold as the label (e.g., a flat "flag every page beyond position 10" rule), rather than one built from the identical ctr/position_tier_ctr comparison.

In [3]:
importance_df = pd.DataFrame({
    'feature': features,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

importance_df

,feature,importance
0,log_impressions,0.961633
1,avg_position,0.038367


Feature importance: log_impressions accounts for 96.2% of the Decision Tree's splits, while avg_position contributes only 3.8% — despite avg_position being the more theoretically meaningful signal in my earlier analysis (ML-02, ML-06, ML-07 all found position-based patterns).

A likely explanation: because my label (underperforming) was defined using a position-tier threshold on ctr, and ctr itself is clicks/impressions, low-impression pages naturally produce noisier, more extreme ctr values (e.g., a page with 1 impression and 0 clicks has ctr=0 exactly, an extreme value). The tree may be using log_impressions as a proxy for "how reliable/extreme is this ctr measurement" rather than avg_position capturing the true underperformance signal directly. This reinforces the Section 4 finding above: my label-baseline coupling issue likely also distorts what the model learns as "important," since impression volume affects how easily the position-tier threshold gets crossed by noise alone, especially for low-volume pages.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.